In [1]:
import sys
sys.path.append("/hpc/group/ydiaolab/yx222/scDeepLUCIA")
from deeplucia_toolkit import make_dataset
from deeplucia_toolkit import misc
import gzip
import itertools
import numpy
import pandas 
import tensorflow 
import matplotlib.pyplot as plt
from sklearn import metrics
from pathlib import Path
import os
os.chdir("/work/yx222/scDeepLUCIA/call_loop")
print(tensorflow.config.list_physical_devices('GPU'))

2026-06-26 14:44:20.555189: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-26 14:44:20.586398: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-26 14:44:20.586428: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-26 14:44:20.586455: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-26 14:44:20.592669: I tensorflow/core/platform/cpu_feature_g

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
resolution = 5000

keras_model_filename = "/hpc/group/ydiaolab/yx222/scDeepLUCIA/model/trained.h5"
chrom = "chr10"
sample = "Meis2"

scan_start = 0
scan_end   = 26137 #check window size from 'scan_window_size' folder

marker_type = "R1"
genome_version = "mm10" 

feature_dirname =  Path.cwd() / "input"
result_dirname =  Path.cwd() / "result" / sample
result_dirname.mkdir(parents=True,exist_ok=True)

prediction_filename = result_dirname / f"score.{chrom}.xls.gz"
loop_bedpe_filename = result_dirname / f"loop.{chrom}.bedpe"

In [3]:
model = misc.load_model(keras_model_filename)

2026-06-26 14:44:28.871225: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 28339 MB memory:  -> device: 0, name: NVIDIA RTX 5000 Ada Generation, pci bus id: 0000:17:00.0, compute capability: 8.9


In [4]:
seq_array_dirname,epi_array_dirname,con_array_dirname = misc.get_directory(feature_dirname,chrom,sample,genome_version)
chrom_sample = (chrom,sample)
chrom_sample_list = [chrom_sample]

chrom_to_seq_array = make_dataset.load_seq_array_dir(chrom_sample_list, seq_array_dirname)
chrom_sample_to_epi_array = make_dataset.load_epi_array_dir(chrom_sample_list , marker_type , epi_array_dirname)
chrom_sample_to_con_array = make_dataset.load_con_array_dir(chrom_sample_list , con_array_dirname)
con_array = chrom_sample_to_con_array[chrom_sample]

Loading sequence array: /work/yx222/scDeepLUCIA/call_loop/input/sliced_seq_array/mm10/seq_onehot.5kb.chr10.npy
Loading epigenomic array: /work/yx222/scDeepLUCIA/call_loop/input/sliced_epi_array/mm10/Meis2/R1.5kb.chr10.npy
Loading contact array: /work/yx222/scDeepLUCIA/call_loop/input/sliced_con_array/mm10/Meis2/convolved_window5.chr10.npy


In [5]:
scanning_loop_candidate_gen = make_dataset.gen_scanning_loop_candidate(chrom,sample,scan_start,scan_end)

In [6]:
pair_list = []
prob_list = []

for _,chunk in itertools.groupby(enumerate(scanning_loop_candidate_gen) , lambda x : x[0]//32768):
    loop_candidate_list = []
    for _,loop_candidate in chunk:
        pair = loop_candidate[2]
        loop_candidate_list.append(loop_candidate)
        pair_list.append(pair)

    batched_feature,_ = make_dataset.extract_seq_epi_dataset_nonshuffle(loop_candidate_list, chrom_to_seq_array, chrom_sample_to_epi_array)
    batched_feature = [x.astype("float32") for x in batched_feature] # prevent error: JIT compilation failed
    output = model.predict(batched_feature)
    batched_prob_pred = numpy.squeeze(output,axis=1)

    for prob in batched_prob_pred:
        prob_list.append(prob)

2026-06-26 14:44:42.871969: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2026-06-26 14:44:42.946235: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-06-26 14:44:43.008934: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


648/648 [==============================] - 4s 6ms/step


In [7]:
if len(pair_list) == len(prob_list):
    with gzip.open(prediction_filename,"wt") as prediction_file:
        prediction_file.write("chrom\tindex_one\tindex_two\tprob\n")
        for pair,prob in zip(pair_list,prob_list):
            prediction_file.write("\t".join(map(str,[chrom, pair[0],pair[1], prob])) + "\n")

In [8]:
loop_df = pandas.read_table(prediction_filename)

In [10]:
dist_filtered_loop_df = misc.filter_by_distance(loop_df)

In [11]:
cutoff_filtered_loop_df = misc.filter_by_quantile(dist_filtered_loop_df,con_array)

In [12]:
clustered_loop_df = misc.form_loop_cluster(cutoff_filtered_loop_df)

In [13]:
misc.save_as_bedpe(clustered_loop_df,chrom,resolution,loop_bedpe_filename)